# Data Explorer: Open-Ended Exploratory Data Analysis

**Data Explorer** is an AI agent built on [NVIDIA NeMo Agent Toolkit (NAT)](https://github.com/NVIDIA/NeMo-Agent-Toolkit) that automatically generates comprehensive Jupyter notebooks with data analysis and visualizations from any tabular dataset.

The agent uses a **ReAct loop** paired with notebook manipulation tools and a vision-language model for plot feedback, allowing it to iteratively build a polished analysis notebook.

---

## Architecture

```
User (dataset + prompt)
        │
        ▼
┌─────────────────┐       ┌──────────────────────┐
│   ReAct Agent   │──────▶│ Notebook Tools        │
│  (GPT-5 mini)   │◀──────│ (create/edit/run)     │
└─────────────────┘       └──────────┬───────────┘
                                     │
                                     ▼
                          ┌──────────────────────┐
                          │  Vision-Language Model │
                          │  (plot → text feedback)│
                          └──────────────────────┘
```

The agent:
1. Reads your dataset and optional prompt
2. Iteratively adds code and markdown cells to a notebook
3. Executes each cell, inspects outputs, and refines visualizations
4. Produces a self-contained Jupyter notebook as the final artifact

## Setup

Make sure your `OPENAI_API_KEY` environment variable is set. Get one at [platform.openai.com](https://platform.openai.com).

In [ ]:
import os, sys
from dotenv import load_dotenv

os.chdir('/app')
if '/app' not in sys.path:
    sys.path.insert(0, '/app')
load_dotenv()

assert os.environ.get('OPENAI_API_KEY'), (
    'OPENAI_API_KEY not set. Get a key at https://platform.openai.com '
    'and set it: export OPENAI_API_KEY=sk-your_key_here'
)
print('API key configured.')

## Run the EDA Agent

We'll use the sample dataset (`sample_data/QS_2025.csv`) and ask the agent to perform an exploratory analysis.

The agent generates a **new notebook** at `notebooks/example.ipynb` with its analysis. You can change the `user_query` and `dataset_path` below to analyze your own data.

In [ ]:
import asyncio
from nat.runtime.loader import load_workflow
from data_explorer_agent.utils import print_token_usage

CONFIG_FILE = 'src/data_explorer_agent/configs/config_launchable.yml'

dataset_path = 'sample_data/QS_2025.csv'
user_query = f"""Based on this dataset in {dataset_path}, perform an exploratory data analysis.
Report the key findings with clear visualizations. Make it a nice Notebook report."""

async def run_eda():
    async with load_workflow(CONFIG_FILE) as workflow:
        async with workflow.run(user_query) as runner:
            return await runner.result()

result = await run_eda()
print('Agent finished.')
print('Agent response:', result[:500] if result else 'No response')

## View the Generated Notebook

The agent has created (or updated) `notebooks/example.ipynb`. Open it in the file browser on the left to see the full analysis with visualizations.

You can also view a summary below:

In [ ]:
import nbformat

nb_path = 'notebooks/example.ipynb'
if os.path.exists(nb_path):
    with open(nb_path) as f:
        nb = nbformat.read(f, as_version=4)
    print(f'Generated notebook: {nb_path}')
    print(f'Total cells: {len(nb.cells)}')
    print()
    for i, cell in enumerate(nb.cells):
        preview = cell.source[:120].replace('\n', ' ')
        print(f'  Cell {i} ({cell.cell_type}): {preview}...')
else:
    print(f'Notebook not yet generated at {nb_path}. Run the cell above first.')

## Token Usage

In [ ]:
print_token_usage('./traces_eda.jsonl')

---

## Try Your Own Data

Upload your own CSV, Parquet, or JSON file to this environment, then modify the cell below to point to it.

In [ ]:
# Uncomment and modify to run on your own data:

# dataset_path = 'your_data.csv'
# user_query = f'Analyze the dataset at {dataset_path}. Focus on trends and correlations.'
#
# result = await run_eda()
# print('Done! Open notebooks/example.ipynb to see the results.')